In [0]:
dbutils.widgets.removeAll()

In [0]:
from datetime import datetime, timezone

dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

dbutils.widgets.text(
    "ingestion_timestamp",
    datetime.now(timezone.utc).isoformat(),
    "Ingestion Timestamp"
)

environment = dbutils.widgets.get("environment").lower()

ingestion_timestamp = (
    dbutils.widgets.get("ingestion_timestamp").strip()
)

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "storage_account": "stcentralusjrdev",
        "catalog": "saleslt_dev"
    },
    "prod": {
        "storage_account": "stcentralusjrprod",
        "catalog": "saleslt_prod"
    }
}

env = config[environment]

storage_account = env["storage_account"]
catalog = env["catalog"]

print("=" * 60)
print("SALESLT - SILVER TRANSFORMATION")
print("=" * 60)
print(f"Environment          : {environment}")
print(f"Ingestion timestamp  : {ingestion_timestamp}")
print(f"Catalog              : {catalog}")
print("=" * 60)

In [0]:
customer_bronze = (
    f"{catalog}.bronze.customer_raw"
)

product_bronze = (
    f"{catalog}.bronze.product_raw"
)

category_bronze = (
    f"{catalog}.bronze.product_category_raw"
)

header_bronze = (
    f"{catalog}.bronze.sales_order_header_raw"
)

detail_bronze = (
    f"{catalog}.bronze.sales_order_detail_raw"
)


customer_silver = (
    f"{catalog}.silver.customers"
)

product_silver = (
    f"{catalog}.silver.products"
)

sales_silver = (
    f"{catalog}.silver.sales_order_lines"
)


customer_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "saleslt/silver/customers/"
)

product_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "saleslt/silver/products/"
)

sales_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "saleslt/silver/sales_order_lines/"
)

In [0]:
customers_df = spark.sql(f"""
SELECT
    CustomerID AS customer_id,

    TRIM(FirstName) AS first_name,

    NULLIF(
        TRIM(MiddleName),
        ''
    ) AS middle_name,

    TRIM(LastName) AS last_name,

    TRIM(
        CONCAT_WS(
            ' ',
            FirstName,
            NULLIF(MiddleName, ''),
            LastName
        )
    ) AS full_name,

    NULLIF(
        TRIM(CompanyName),
        ''
    ) AS company_name,

    LOWER(
        TRIM(EmailAddress)
    ) AS email_address,

    NULLIF(
        TRIM(Phone),
        ''
    ) AS phone,

    ModifiedDate AS source_modified_timestamp,

    CAST(
        '{ingestion_timestamp}'
        AS TIMESTAMP
    ) AS silver_processing_timestamp

FROM {customer_bronze}

WHERE CustomerID IS NOT NULL
""")

In [0]:
products_df = spark.sql(f"""
SELECT
    p.ProductID AS product_id,

    TRIM(
        p.Name
    ) AS product_name,

    TRIM(
        p.ProductNumber
    ) AS product_number,

    NULLIF(
        TRIM(p.Color),
        ''
    ) AS color,

    CAST(
        p.StandardCost
        AS DECIMAL(18,2)
    ) AS standard_cost,

    CAST(
        p.ListPrice
        AS DECIMAL(18,2)
    ) AS list_price,

    NULLIF(
        TRIM(p.Size),
        ''
    ) AS size,

    CAST(
        p.Weight
        AS DECIMAL(18,2)
    ) AS weight,

    p.ProductCategoryID AS product_category_id,

    TRIM(
        pc.Name
    ) AS product_category,

    p.SellStartDate AS sell_start_date,

    p.SellEndDate AS sell_end_date,

    p.DiscontinuedDate AS discontinued_date,

    CASE
        WHEN p.DiscontinuedDate IS NULL
            THEN true
        ELSE false
    END AS is_active,

    p.ModifiedDate AS source_modified_timestamp,

    CAST(
        '{ingestion_timestamp}'
        AS TIMESTAMP
    ) AS silver_processing_timestamp

FROM {product_bronze} p

LEFT JOIN {category_bronze} pc
    ON p.ProductCategoryID = pc.ProductCategoryID

WHERE p.ProductID IS NOT NULL
""")

In [0]:
sales_order_lines_df = spark.sql(f"""
SELECT
    d.SalesOrderDetailID
        AS sales_order_detail_id,

    h.SalesOrderID
        AS sales_order_id,

    h.SalesOrderNumber
        AS sales_order_number,

    h.CustomerID
        AS customer_id,

    CONCAT_WS(
        ' ',
        c.FirstName,
        NULLIF(c.MiddleName, ''),
        c.LastName
    ) AS customer_name,

    LOWER(
        TRIM(c.EmailAddress)
    ) AS customer_email,

    d.ProductID
        AS product_id,

    TRIM(
        p.Name
    ) AS product_name,

    TRIM(
        pc.Name
    ) AS product_category,

    h.OrderDate
        AS order_date,

    h.DueDate
        AS due_date,

    h.ShipDate
        AS ship_date,

    h.Status
        AS order_status,

    h.OnlineOrderFlag
        AS online_order_flag,

    TRIM(
        h.ShipMethod
    ) AS ship_method,

    CAST(
        d.OrderQty
        AS INT
    ) AS order_quantity,

    CAST(
        d.UnitPrice
        AS DECIMAL(18,2)
    ) AS unit_price,

    CAST(
        d.UnitPriceDiscount
        AS DECIMAL(10,4)
    ) AS unit_price_discount,

    CAST(
        d.OrderQty * d.UnitPrice
        AS DECIMAL(18,2)
    ) AS gross_line_amount,

    CAST(
        (
            d.OrderQty
            * d.UnitPrice
            * d.UnitPriceDiscount
        )
        AS DECIMAL(18,2)
    ) AS discount_amount,

    CAST(
        d.LineTotal
        AS DECIMAL(18,2)
    ) AS net_line_amount,

    CAST(
        h.SubTotal
        AS DECIMAL(18,2)
    ) AS order_subtotal,

    CAST(
        h.TaxAmt
        AS DECIMAL(18,2)
    ) AS order_tax_amount,

    CAST(
        h.Freight
        AS DECIMAL(18,2)
    ) AS order_freight_amount,

    CAST(
        h.TotalDue
        AS DECIMAL(18,2)
    ) AS order_total_due,

    d.ModifiedDate
        AS source_modified_timestamp,

    CAST(
        '{ingestion_timestamp}'
        AS TIMESTAMP
    ) AS silver_processing_timestamp

FROM {detail_bronze} d

INNER JOIN {header_bronze} h
    ON d.SalesOrderID = h.SalesOrderID

INNER JOIN {customer_bronze} c
    ON h.CustomerID = c.CustomerID

INNER JOIN {product_bronze} p
    ON d.ProductID = p.ProductID

LEFT JOIN {category_bronze} pc
    ON p.ProductCategoryID = pc.ProductCategoryID

WHERE
    d.SalesOrderDetailID IS NOT NULL

    AND d.OrderQty > 0

    AND d.UnitPrice >= 0

    AND d.UnitPriceDiscount >= 0

    AND d.UnitPriceDiscount <= 1
""")

In [0]:
# COMMAND ----------

from delta.tables import DeltaTable


def merge_silver_snapshot(
    source_df,
    target_table,
    target_path,
    merge_key
):
    """
    Creates an external Silver Delta table on initial load.

    Subsequent executions synchronize the target with the
    latest transformed Bronze snapshot.
    """

    if not spark.catalog.tableExists(target_table):

        print(
            f"Initial load. Creating Silver table: "
            f"{target_table}"
        )

        (
            source_df.write
                .format("delta")
                .mode("append")
                .option(
                    "mergeSchema",
                    "true"
                )
                .option(
                    "path",
                    target_path
                )
                .saveAsTable(
                    target_table
                )
        )

    else:

        print(
            f"Synchronizing Silver table: "
            f"{target_table}"
        )

        target = DeltaTable.forName(
            spark,
            target_table
        )

        (
            target.alias("target")

                .merge(
                    source_df.alias("source"),
                    f"""
                    target.{merge_key}
                    =
                    source.{merge_key}
                    """
                )

                .withSchemaEvolution()

                .whenMatchedUpdateAll()

                .whenNotMatchedInsertAll()

                .whenNotMatchedBySourceDelete()

                .execute()
        )

    print(
        f"Silver synchronization completed: "
        f"{target_table}"
    )

In [0]:
merge_silver_snapshot(
    source_df=customers_df,
    target_table=customer_silver,
    target_path=customer_path,
    merge_key="customer_id"
)


merge_silver_snapshot(
    source_df=products_df,
    target_table=product_silver,
    target_path=product_path,
    merge_key="product_id"
)

merge_silver_snapshot(
    source_df=sales_order_lines_df,
    target_table=sales_silver,
    target_path=sales_path,
    merge_key="sales_order_detail_id"
)